# PS S6E6 - 005 OOF correlation check for raw baselines

Experiment: `exp_20260603_005_oof_corr_raw_baselines`

Purpose:
- Compare OOF predictions from 002/003/004 strict raw baselines.
- Check whether CatBoost/XGBoost are just weaker copies of LightGBM or potentially useful auxiliary materials.
- Compute individual CV, pairwise OOF correlations, disagreement, error overlap, and simple average ensemble diagnostics.
- Save correlation matrices, pairwise summary, ensemble diagnostics, and memo.

Input models:
- `002`: LightGBM strict raw baseline
- `003`: CatBoost strict raw baseline
- `004`: XGBoost strict raw baseline

Metric:
- `balanced_accuracy_score`

Date:
- 2026-06-03

In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import os
import json
import math
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import rankdata, spearmanr
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None


COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260603_005_oof_corr_raw_baselines"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
NPY_PATH: str = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "class"
ID_COL = "id"

# These paths are the default paths if 002/003/004 outputs are available in the same working area.
# If previous notebook outputs were added as Kaggle input datasets, resolve_path() will also search /kaggle/input recursively by filename.
MODEL_SPECS = [
    {
        "key": "002_lgb_raw",
        "exp_id": "exp_20260601_002_lgb_strict_raw_baseline",
        "family": "LightGBM",
        "oof": NPY_PATH + "oof_lgb_strict_raw_proba.npy",
        "pred": NPY_PATH + "pred_lgb_strict_raw_proba.npy",
        "cv": 0.9569063401002502,
        "public_lb": 0.95800,
    },
    {
        "key": "003_cat_raw",
        "exp_id": "exp_20260601_003_cat_strict_raw_baseline",
        "family": "CatBoost",
        "oof": NPY_PATH + "oof_cat_strict_raw_proba.npy",
        "pred": NPY_PATH + "pred_cat_strict_raw_proba.npy",
        "cv": 0.9524581895303758,
        "public_lb": 0.95351,
    },
    {
        "key": "004_xgb_raw",
        "exp_id": "exp_20260603_004_xgb_strict_raw_baseline",
        "family": "XGBoost",
        "oof": NPY_PATH + "oof_xgb_strict_raw_proba.npy",
        "pred": NPY_PATH + "pred_xgb_strict_raw_proba.npy",
        "cv": 0.9557836270283273,
        "public_lb": 0.95638,
    },
]

print("OUTDIR:", OUTDIR)

OUTDIR: /kaggle/working/exp_20260603_005_oof_corr_raw_baselines


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def resolve_path(path_like: str) -> Path:
    """Resolve an OOF/pred path.

    Search order:
    1. exact path
    2. /kaggle/working/**/filename
    3. /kaggle/input/**/filename

    This makes the notebook usable both in the same session and with previous notebook outputs added as datasets.
    """
    p = Path(path_like)
    if p.exists():
        return p

    fname = p.name
    search_roots = [Path("/kaggle/working"), Path("/kaggle/input")]

    matches = []
    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(fname))

    matches = sorted(set(matches))
    if len(matches) == 1:
        print(f"[resolve_path] {path_like} -> {matches[0]}")
        return matches[0]
    if len(matches) > 1:
        print(f"[resolve_path] Multiple candidates for {fname}:")
        for m in matches[:20]:
            print(" -", m)
        print("[resolve_path] Using first candidate:", matches[0])
        return matches[0]

    raise FileNotFoundError(
        f"Could not resolve path: {path_like}\\n"
        f"Expected filename: {fname}\\n"
        "If this happens, add previous notebook outputs as Kaggle input datasets or edit MODEL_SPECS paths."
    )


def proba_to_pred(proba: np.ndarray) -> np.ndarray:
    return np.argmax(proba, axis=1)


def flatten_proba(proba: np.ndarray) -> np.ndarray:
    return np.asarray(proba, dtype=float).reshape(-1)


def corr_pearson(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def corr_spearman(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)


def class_recalls(y_true: np.ndarray, y_pred: np.ndarray, class_names: list[str]) -> dict:
    recalls = {}
    for i, cls in enumerate(class_names):
        mask = y_true == i
        recalls[f"recall_{cls}"] = float((y_pred[mask] == i).mean()) if mask.any() else float("nan")
    return recalls


def ensemble_average(keys: list[str], oof_dict: dict[str, np.ndarray], pred_dict: dict[str, np.ndarray] | None = None):
    oof_avg = np.mean([oof_dict[k] for k in keys], axis=0)
    pred_avg = None
    if pred_dict is not None and all(k in pred_dict for k in keys):
        pred_avg = np.mean([pred_dict[k] for k in keys], axis=0)
    return oof_avg, pred_avg

In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

if not TRAIN_PATH.exists():
    raise FileNotFoundError(TRAIN_PATH)

train = pd.read_csv(TRAIN_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)

print("train:", train.shape)
print("class_names:", class_names)

oof = {}
pred = {}
resolved_specs = []

for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = resolve_path(spec["oof"])
    oof_arr = np.load(oof_path)

    if oof_arr.shape != (len(train), n_classes):
        raise ValueError(f"{key} OOF shape mismatch: {oof_arr.shape}, expected {(len(train), n_classes)}")

    oof[key] = oof_arr.astype(np.float32)

    pred_path = None
    pred_arr = None
    try:
        pred_path = resolve_path(spec["pred"])
        pred_arr = np.load(pred_path).astype(np.float32)
        pred[key] = pred_arr
    except Exception as e:
        print(f"[WARN] pred not loaded for {key}: {e}")

    resolved = dict(spec)
    resolved["resolved_oof_path"] = str(oof_path)
    resolved["resolved_pred_path"] = str(pred_path) if pred_path is not None else None
    resolved["oof_shape"] = list(oof_arr.shape)
    resolved["pred_shape"] = list(pred_arr.shape) if pred_arr is not None else None
    resolved_specs.append(resolved)

model_keys = [s["key"] for s in MODEL_SPECS]
print("loaded model_keys:", model_keys)
pd.DataFrame(resolved_specs)

train: (577347, 12)
class_names: ['GALAXY', 'QSO', 'STAR']
loaded model_keys: ['002_lgb_raw', '003_cat_raw', '004_xgb_raw']


,key,exp_id,family,oof,pred,cv,public_lb,resolved_oof_path,resolved_pred_path,oof_shape,pred_shape
0,002_lgb_raw,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,0.956906,0.95800,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
1,003_cat_raw,exp_20260601_003_cat_strict_raw_baseline,CatBoost,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,0.952458,0.95351,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
2,004_xgb_raw,exp_20260603_004_xgb_strict_raw_baseline,XGBoost,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,0.955784,0.95638,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"


In [4]:
# ============================================================
# 3. Individual scores
# ============================================================

individual_rows = []

for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    pred_label = proba_to_pred(p)
    score = balanced_accuracy_score(y, pred_label)

    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "cv_from_json": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_json": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, pred_label, class_names))
    individual_rows.append(row)

individual_df = pd.DataFrame(individual_rows).sort_values("recomputed_oof_cv", ascending=False)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

,key,exp_id,family,cv_from_json,public_lb,recomputed_oof_cv,delta_recomputed_minus_json,recall_GALAXY,recall_QSO,recall_STAR
0,002_lgb_raw,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,0.956906,0.95800,0.956906,0.0,0.978431,0.965000,0.927288
2,004_xgb_raw,exp_20260603_004_xgb_strict_raw_baseline,XGBoost,0.955784,0.95638,0.955784,0.0,0.978582,0.963318,0.925451
1,003_cat_raw,exp_20260601_003_cat_strict_raw_baseline,CatBoost,0.952458,0.95351,0.952458,0.0,0.977432,0.963498,0.916445


In [5]:
# ============================================================
# 4. Pairwise OOF correlation / disagreement / error overlap
# ============================================================

pair_rows = []

for a, b in combinations(model_keys, 2):
    pa = oof[a]
    pb = oof[b]

    ya = proba_to_pred(pa)
    yb = proba_to_pred(pb)

    wrong_a = ya != y
    wrong_b = yb != y
    both_wrong = wrong_a & wrong_b
    either_wrong = wrong_a | wrong_b

    row = {
        "model_a": a,
        "model_b": b,
        "pearson_flat_proba": corr_pearson(flatten_proba(pa), flatten_proba(pb)),
        "spearman_flat_proba": corr_spearman(flatten_proba(pa), flatten_proba(pb)),
        "argmax_agreement": float((ya == yb).mean()),
        "argmax_disagreement": float((ya != yb).mean()),
        "wrong_rate_a": float(wrong_a.mean()),
        "wrong_rate_b": float(wrong_b.mean()),
        "both_wrong_rate": float(both_wrong.mean()),
        "either_wrong_rate": float(either_wrong.mean()),
        "error_jaccard": float(both_wrong.sum() / max(either_wrong.sum(), 1)),
        "a_wrong_b_correct_rate": float((wrong_a & ~wrong_b).mean()),
        "a_correct_b_wrong_rate": float((~wrong_a & wrong_b).mean()),
    }

    for ci, cls in enumerate(class_names):
        row[f"pearson_proba_{cls}"] = corr_pearson(pa[:, ci], pb[:, ci])
        row[f"spearman_proba_{cls}"] = corr_spearman(pa[:, ci], pb[:, ci])

    pair_rows.append(row)

pairwise_df = pd.DataFrame(pair_rows).sort_values("pearson_flat_proba")
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

,model_a,model_b,pearson_flat_proba,spearman_flat_proba,argmax_agreement,argmax_disagreement,wrong_rate_a,wrong_rate_b,both_wrong_rate,either_wrong_rate,error_jaccard,a_wrong_b_correct_rate,a_correct_b_wrong_rate,pearson_proba_GALAXY,spearman_proba_GALAXY,pearson_proba_QSO,spearman_proba_QSO,pearson_proba_STAR,spearman_proba_STAR
0,002_lgb_raw,003_cat_raw,0.996519,0.981796,0.989538,0.010462,0.031622,0.034134,0.027735,0.038020,0.729488,0.003887,0.006398,0.995920,0.982472,0.996843,0.939371,0.992158,0.928582
2,003_cat_raw,004_xgb_raw,0.997419,0.981125,0.990425,0.009575,0.034134,0.032128,0.028434,0.037828,0.751648,0.005700,0.003694,0.996561,0.982859,0.998087,0.946916,0.994371,0.904603
1,002_lgb_raw,004_xgb_raw,0.997944,0.989543,0.992682,0.007318,0.031622,0.032128,0.028298,0.035452,0.798222,0.003324,0.003830,0.997946,0.990355,0.997615,0.964777,0.995412,0.943056


In [6]:
# ============================================================
# 5. Correlation matrices
# ============================================================

pearson_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
spearman_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
argmax_agreement = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
error_jaccard = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)

pred_labels = {k: proba_to_pred(oof[k]) for k in model_keys}
wrong_flags = {k: pred_labels[k] != y for k in model_keys}

for a in model_keys:
    for b in model_keys:
        pearson_flat.loc[a, b] = corr_pearson(flatten_proba(oof[a]), flatten_proba(oof[b]))
        spearman_flat.loc[a, b] = corr_spearman(flatten_proba(oof[a]), flatten_proba(oof[b]))
        argmax_agreement.loc[a, b] = float((pred_labels[a] == pred_labels[b]).mean())
        both = wrong_flags[a] & wrong_flags[b]
        either = wrong_flags[a] | wrong_flags[b]
        error_jaccard.loc[a, b] = float(both.sum() / max(either.sum(), 1))

display(pearson_flat)
display(spearman_flat)
display(argmax_agreement)
display(error_jaccard)

pearson_flat.to_csv(OUTDIR / "corr_matrix_pearson_flat_proba.csv")
spearman_flat.to_csv(OUTDIR / "corr_matrix_spearman_flat_proba.csv")
argmax_agreement.to_csv(OUTDIR / "matrix_argmax_agreement.csv")
error_jaccard.to_csv(OUTDIR / "matrix_error_jaccard.csv")

,002_lgb_raw,003_cat_raw,004_xgb_raw
002_lgb_raw,1.000000,0.996519,0.997944
003_cat_raw,0.996519,1.000000,0.997419
004_xgb_raw,0.997944,0.997419,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw
002_lgb_raw,1.000000,0.981796,0.989543
003_cat_raw,0.981796,1.000000,0.981125
004_xgb_raw,0.989543,0.981125,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw
002_lgb_raw,1.000000,0.989538,0.992682
003_cat_raw,0.989538,1.000000,0.990425
004_xgb_raw,0.992682,0.990425,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw
002_lgb_raw,1.000000,0.729488,0.798222
003_cat_raw,0.729488,1.000000,0.751648
004_xgb_raw,0.798222,0.751648,1.000000


In [7]:
# ============================================================
# 6. Simple average ensemble diagnostics
# ============================================================

ensemble_specs = [
    ["002_lgb_raw", "003_cat_raw"],
    ["002_lgb_raw", "004_xgb_raw"],
    ["003_cat_raw", "004_xgb_raw"],
    ["002_lgb_raw", "003_cat_raw", "004_xgb_raw"],
]

ensemble_rows = []

for keys in ensemble_specs:
    if not all(k in oof for k in keys):
        continue

    ens_oof, _ = ensemble_average(keys, oof, pred)
    ens_pred = proba_to_pred(ens_oof)
    score = balanced_accuracy_score(y, ens_pred)

    row = {
        "strategy": "avg_" + "_".join(keys),
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": float(score),
    }
    row.update(class_recalls(y, ens_pred, class_names))
    ensemble_rows.append(row)

ensemble_df = pd.DataFrame(ensemble_rows).sort_values("balanced_accuracy", ascending=False)
display(ensemble_df)
ensemble_df.to_csv(OUTDIR / "simple_average_ensemble_diagnostics.csv", index=False)

,strategy,keys,n_models,balanced_accuracy,recall_GALAXY,recall_QSO,recall_STAR
1,avg_002_lgb_raw_004_xgb_raw,"002_lgb_raw,004_xgb_raw",2,0.957038,0.978725,0.964642,0.927748
3,avg_002_lgb_raw_003_cat_raw_004_xgb_raw,"002_lgb_raw,003_cat_raw,004_xgb_raw",3,0.956213,0.978560,0.964471,0.925608
0,avg_002_lgb_raw_003_cat_raw,"002_lgb_raw,003_cat_raw",2,0.956128,0.978412,0.964932,0.925040
2,avg_003_cat_raw_004_xgb_raw,"003_cat_raw,004_xgb_raw",2,0.954956,0.978356,0.963865,0.922646


In [8]:
# ============================================================
# 7. Rank / role judgement
# ============================================================

# Mechanical first judgement from raw CV/LB and pairwise correlation.
# Final judgement should be edited after reviewing Public LB and future FE/bias experiments.

best_single_key = individual_df.iloc[0]["key"]
best_single_cv = float(individual_df.iloc[0]["recomputed_oof_cv"])

role_rows = []
for _, row in individual_df.iterrows():
    key = row["key"]
    cv = float(row["recomputed_oof_cv"])
    lb = row["public_lb"]

    if key == "002_lgb_raw":
        role = "main_axis_candidate"
        judgement = "keep_main_reference"
    elif key == "004_xgb_raw":
        role = "auxiliary_candidate"
        judgement = "keep_aux_reference"
    elif key == "003_cat_raw":
        role = "reference_or_low_corr_candidate"
        judgement = "keep_reference"
    else:
        role = "unknown"
        judgement = "hold"

    role_rows.append({
        "key": key,
        "cv": cv,
        "public_lb": lb,
        "role": role,
        "judgement": judgement,
    })

role_df = pd.DataFrame(role_rows)
display(role_df)
role_df.to_csv(OUTDIR / "role_judgement.csv", index=False)

,key,cv,public_lb,role,judgement
0,002_lgb_raw,0.956906,0.95800,main_axis_candidate,keep_main_reference
1,004_xgb_raw,0.955784,0.95638,auxiliary_candidate,keep_aux_reference
2,003_cat_raw,0.952458,0.95351,reference_or_low_corr_candidate,keep_reference


In [9]:
# ============================================================
# 8. Save summary / memo
# ============================================================

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "OOF correlation check for 002/003/004 strict raw baselines",
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "simple_average_ensemble_diagnostics": ensemble_df.to_dict(orient="records"),
    "role_judgement": role_df.to_dict(orient="records"),
}

save_json(summary, OUTDIR / "oof_corr_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "OOF correlation check for 002/003/004 raw baselines",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-03",
    },
    "objective": {
        "summary": (
            "Compare OOF predictions from LightGBM/CatBoost/XGBoost strict raw baselines. "
            "Compute individual CV, pairwise proba correlation, argmax disagreement, error overlap, "
            "and simple average ensemble diagnostics before moving to FE/original/bias experiments."
        ),
        "success_criteria": [
            "load OOF proba for 002/003/004",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations",
            "compute argmax disagreement and error overlap",
            "compute simple average ensemble diagnostics",
            "save summary files",
        ],
    },
    "inputs": {
        "models": resolved_specs,
    },
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "corr_matrix_pearson_flat_proba": "corr_matrix_pearson_flat_proba.csv",
        "corr_matrix_spearman_flat_proba": "corr_matrix_spearman_flat_proba.csv",
        "matrix_argmax_agreement": "matrix_argmax_agreement.csv",
        "matrix_error_jaccard": "matrix_error_jaccard.csv",
        "simple_average_ensemble_diagnostics": "simple_average_ensemble_diagnostics.csv",
        "role_judgement": "role_judgement.csv",
        "summary": "oof_corr_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "Based on prior CV/LB, 002 LightGBM is the main raw baseline, "
            "004 XGBoost is the next auxiliary candidate, and 003 CatBoost is reference/possible low-correlation material. "
            "Use the generated correlation files to decide whether Cat/XGB should be kept for later blend."
        ),
        "next_action": [
            "Review pairwise_oof_correlation.csv",
            "Review simple_average_ensemble_diagnostics.csv",
            "If 002+004 average improves OOF, keep XGB as early blend material",
            "Do not start Optuna yet",
            "Next FE candidate should likely be color index minimum on LGB/XGB after this review",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved files:
 - corr_matrix_pearson_flat_proba.csv
 - corr_matrix_spearman_flat_proba.csv
 - individual_scores.csv
 - matrix_argmax_agreement.csv
 - matrix_error_jaccard.csv
 - memo.yml
 - oof_corr_summary.json
 - pairwise_oof_correlation.csv
 - role_judgement.csv
 - simple_average_ensemble_diagnostics.csv
